# OgmahDemo — Exploration & Choix des Modèles

**Contexte** : ogmah est un SaaS de gestion de restaurant. Ce notebook justifie les choix techniques pour les trois modules IA :

1. **Détection d'anomalies** sur les prix d'achat fournisseurs
2. **Prévision de demande** par recette
3. **Analyse des marges** et opportunités d'optimisation

Les données sont synthétiques mais représentatives d'un restaurant 80-couverts avec 10 plats en carte.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import date, timedelta
from scipy import stats

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

print('Librairies chargées.')

## 1. Génération des données synthétiques

On reproduit la logique du script `seed.py` directement en mémoire pour un EDA standalone.

In [ ]:
# ── Paramètres ─────────────────────────────────────────────────────────────
END_DATE   = date.today()
START_DATE = END_DATE - timedelta(days=180)
DATES      = pd.date_range(START_DATE, END_DATE, freq='D')

INGREDIENTS = [
    ('Bœuf entrecôte',  'kg',  'viande',     'Maison Dupont',        22.0),
    ('Poulet fermier',  'kg',  'viande',     'Volailles Martin',      8.5),
    ('Saumon frais',    'kg',  'poisson',    'Marée Fraîche',        18.0),
    ('Pommes de terre', 'kg',  'légume',     'Primeur Local',         0.8),
    ('Crème fraîche',   'L',   'laitier',    'Fromagerie Bertrand',   3.2),
    ('Beurre',          'kg',  'laitier',    'Fromagerie Bertrand',   9.0),
    ('Vin rouge',       'L',   'boisson',    'Cave du Patron',        6.0),
    ('Champignons',     'kg',  'légume',     'Champignonnière Bio',   5.5),
    ('Lardons',         'kg',  'charcuterie','Charcuterie Renard',    7.0),
    ('Parmesan',        'kg',  'laitier',    'Fromagerie Bertrand',  18.0),
    ('Pâtes fraîches',  'kg',  'épicerie',   'Pasta Nobile',          4.5),
    ('Tomates',         'kg',  'légume',     'Primeur Local',         2.2),
    ('Huile d\'olive', 'L',   'épicerie',   'Oliviers & Co',         8.5),
    ('Oignons',         'kg',  'légume',     'Primeur Local',         0.6),
    ('Ail',             'kg',  'légume',     'Primeur Local',         4.0),
]

RECIPES = [
    ('Entrecôte bordelaise',  'plat',          28.0, 12),
    ('Poulet rôti',           'plat',          18.5, 18),
    ('Pavé de saumon',        'plat',          22.0, 15),
    ('Tagliatelles carbonara','plat',          14.0, 22),
    ('Bœuf bourguignon',      'plat',          24.0, 10),
    ('Gratin dauphinois',     'accompagnement', 6.5, 25),
    ('Salade tomates',        'entrée',         9.0, 20),
    ('Pasta al pomodoro',     'plat',          12.0, 16),
    ('Risotto champignons',   'plat',          16.0, 11),
    ('Poulet basquaise',      'plat',          19.0, 14),
]

# Food cost réels approximatifs (coût ingrédients / prix vente)
FOOD_COST = {
    'Entrecôte bordelaise':   0.34,
    'Poulet rôti':            0.22,
    'Pavé de saumon':         0.29,
    'Tagliatelles carbonara': 0.18,
    'Bœuf bourguignon':       0.38,
    'Gratin dauphinois':      0.14,
    'Salade tomates':         0.20,
    'Pasta al pomodoro':      0.19,
    'Risotto champignons':    0.26,
    'Poulet basquaise':       0.24,
}

print(f'Période : {START_DATE} → {END_DATE}  ({len(DATES)} jours)')
print(f'{len(INGREDIENTS)} ingrédients, {len(RECIPES)} recettes')

In [ ]:
# ── Génération des achats ──────────────────────────────────────────────────
purchase_rows = []
for name, unit, cat, supplier, base_price in INGREDIENTS:
    for i, d in enumerate(DATES):
        if d.dayofweek not in (0, 2, 4):   # livraisons lun/mer/ven
            continue
        if np.random.random() > 0.70:
            continue
        drift   = 1 + 0.05 * i / len(DATES)
        noise   = np.random.normal(1.0, 0.04)
        anomaly = np.random.random() < 0.03
        if anomaly:
            noise = np.random.uniform(1.25, 1.60)
        price = base_price * drift * noise
        purchase_rows.append({
            'ingredient': name, 'category': cat, 'supplier': supplier,
            'date': d, 'unit_price': round(price, 4),
            'base_price': base_price, 'is_anomaly': anomaly
        })

purchases = pd.DataFrame(purchase_rows)
purchases['date'] = pd.to_datetime(purchases['date'])

# ── Génération des ventes ──────────────────────────────────────────────────
sale_rows = []
for d in DATES:
    weekend = 1.4 if d.dayofweek >= 4 else 1.0
    summer  = 1.2 if d.month in (6, 7, 8) else 1.0
    for name, cat, price, base_qty in RECIPES:
        qty = max(1, int(base_qty * weekend * summer * np.random.normal(1.0, 0.15)))
        fc  = FOOD_COST[name]
        sale_rows.append({
            'recipe': name, 'category': cat, 'selling_price': price,
            'date': d, 'qty': qty, 'revenue': qty * price,
            'ingredient_cost': round(price * fc, 4),
            'food_cost_ratio': fc,
        })

sales = pd.DataFrame(sale_rows)
sales['date'] = pd.to_datetime(sales['date'])

print(f'Achats : {len(purchases):,} lignes   |   dont anomalies injectées : {purchases.is_anomaly.sum()}')
print(f'Ventes : {len(sales):,} lignes')

## 2. EDA — Exploration des données

In [ ]:
# ── Ventes journalières globales ───────────────────────────────────────────
daily = sales.groupby('date').agg(revenue=('revenue','sum'), covers=('qty','sum')).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(daily['date'], daily['revenue'].rolling(7).mean(), color='steelblue', lw=2)
axes[0].fill_between(daily['date'], daily['revenue'], alpha=0.15, color='steelblue')
axes[0].set_title('CA journalier (moyenne mobile 7j)', fontweight='bold')
axes[0].set_ylabel('€')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

axes[1].plot(daily['date'], daily['covers'].rolling(7).mean(), color='coral', lw=2)
axes[1].fill_between(daily['date'], daily['covers'], alpha=0.15, color='coral')
axes[1].set_title('Couverts journaliers (moyenne mobile 7j)', fontweight='bold')
axes[1].set_ylabel('Portions')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.show()

print(f"CA moyen/jour : {daily['revenue'].mean():.0f} €  |  "
      f"CA total 6 mois : {daily['revenue'].sum():,.0f} €")

In [ ]:
# ── Top recettes par volume de vente ──────────────────────────────────────
top_recipes = sales.groupby('recipe').agg(
    total_qty=('qty','sum'),
    total_revenue=('revenue','sum'),
    food_cost_ratio=('food_cost_ratio','first'),
).sort_values('total_qty', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#ef4444' if r > 0.30 else '#22c55e'
          for r in top_recipes['food_cost_ratio']]

axes[0].barh(top_recipes.index, top_recipes['total_qty'], color='steelblue', alpha=0.8)
axes[0].set_title('Volume de ventes (6 mois)', fontweight='bold')
axes[0].set_xlabel('Portions vendues')

axes[1].barh(top_recipes.index, top_recipes['food_cost_ratio'] * 100, color=colors, alpha=0.85)
axes[1].axvline(30, color='black', linestyle='--', lw=1.5, label='Objectif 30%')
axes[1].set_title('Food Cost Ratio par recette', fontweight='bold')
axes[1].set_xlabel('Food Cost %')
axes[1].legend()

plt.tight_layout()
plt.show()

above = (top_recipes['food_cost_ratio'] > 0.30).sum()
print(f'{above}/{len(top_recipes)} recettes dépassent l\'objectif de 30% de food cost')

In [ ]:
# ── Dérive des prix d'achat dans le temps ─────────────────────────────────
focus = ['Bœuf entrecôte', 'Saumon frais', 'Pommes de terre', 'Parmesan']
fig, axes = plt.subplots(2, 2, figsize=(14, 7))

for ax, ing in zip(axes.flat, focus):
    df = purchases[purchases['ingredient'] == ing].sort_values('date')
    ax.scatter(df['date'], df['unit_price'],
               c=['#ef4444' if a else '#94a3b8' for a in df['is_anomaly']],
               s=18, alpha=0.7, zorder=3)
    rolling = df.set_index('date')['unit_price'].rolling('30D').mean()
    ax.plot(rolling.index, rolling.values, color='steelblue', lw=2, label='Moy. 30j')
    ax.set_title(ing, fontweight='bold')
    ax.set_ylabel('€/unité')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    anomalies = df[df['is_anomaly']]
    if not anomalies.empty:
        ax.scatter(anomalies['date'], anomalies['unit_price'],
                   color='#ef4444', s=60, zorder=4, label=f'{len(anomalies)} anomalies')
    ax.legend(fontsize=9)

plt.suptitle('Évolution des prix d\'achat — points rouges = anomalies injectées',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Module 1 — Détection d'anomalies sur les prix fournisseurs

### Choix du modèle

| Méthode | Avantages | Limites | Adapté ici ? |
|---|---|---|---|
| Z-score | Simple, rapide | Assume gaussien, sensible aux outliers de la baseline | Moyen |
| IQR | Robuste, non-paramétrique | Pas de notion temporelle | Moyen |
| **Isolation Forest** | Non-paramétrique, gère la dérive temporelle, score de confiance | Moins interprétable | **✓ Retenu** |

**Décision** : Isolation Forest sur le log-ratio `prix_achat / moyenne_30j`. Le log-ratio est scale-invariant (un bœuf à 22€/kg et du sel à 0.5€/kg sont comparables).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Feature engineering : log-ratio vs moyenne mobile 30j
purch = purchases.sort_values(['ingredient', 'date']).copy()
purch['rolling_avg'] = (
    purch.groupby('ingredient')['unit_price']
         .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
         .fillna(purch['unit_price'])
)
purch['price_ratio'] = purch['unit_price'] / purch['rolling_avg']
purch['log_ratio']   = np.log(purch['price_ratio'].clip(lower=0.01))
purch['deviation_pct'] = (purch['price_ratio'] - 1) * 100

# Comparaison Z-score vs IQR vs Isolation Forest
X = purch[['log_ratio']].values

# Z-score
z_scores = np.abs(stats.zscore(X.flatten()))
purch['zscore_flag']  = z_scores > 3

# IQR
Q1, Q3 = np.percentile(X, [25, 75])
IQR = Q3 - Q1
purch['iqr_flag'] = (X.flatten() < Q1 - 1.5*IQR) | (X.flatten() > Q3 + 1.5*IQR)

# Isolation Forest
iforest = IsolationForest(contamination=0.03, n_estimators=100, random_state=42)
purch['if_pred']  = iforest.fit_predict(X)
purch['if_flag']  = purch['if_pred'] == -1
purch['if_score'] = iforest.decision_function(X)

# Résultats
print('── Performances sur les anomalies injectées (ground truth) ──')
for method, col in [('Z-score (>3σ)', 'zscore_flag'), ('IQR (1.5×)', 'iqr_flag'), ('Isolation Forest', 'if_flag')]:
    tp = (purch['is_anomaly'] & purch[col]).sum()
    fp = (~purch['is_anomaly'] & purch[col]).sum()
    fn = (purch['is_anomaly'] & ~purch[col]).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    print(f'  {method:20s}  Précision={prec:.2f}  Rappel={rec:.2f}  F1={f1:.2f}  (FP={fp})')

In [ ]:
# ── Visualisation : distribution des log-ratios ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(purch[~purch['is_anomaly']]['log_ratio'], bins=60,
             alpha=0.7, label='Normal', color='steelblue')
axes[0].hist(purch[purch['is_anomaly']]['log_ratio'], bins=20,
             alpha=0.85, label='Anomalie (injectée)', color='#ef4444')
axes[0].set_title('Distribution des log-ratios prix/moyenne-30j', fontweight='bold')
axes[0].set_xlabel('log(prix / moy_30j)')
axes[0].legend()

# Score Isolation Forest
threshold = np.percentile(purch['if_score'], 3)
axes[1].scatter(purch['deviation_pct'], purch['if_score'],
                c=['#ef4444' if f else '#94a3b8' for f in purch['if_flag']],
                s=10, alpha=0.5)
axes[1].axhline(threshold, color='black', lw=1.5, linestyle='--', label='Seuil de décision')
axes[1].set_xlabel('Écart au prix moyen (%)')
axes[1].set_ylabel('Score Isolation Forest (bas = plus suspect)')
axes[1].set_title('Score IF vs écart de prix', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Top anomalies détectées ────────────────────────────────────────────────
top_anomalies = (
    purch[purch['if_flag']]
    .sort_values('deviation_pct', ascending=False)
    [['ingredient', 'supplier', 'date', 'unit_price', 'rolling_avg', 'deviation_pct']]
    .head(10)
)
top_anomalies['deviation_pct'] = top_anomalies['deviation_pct'].round(1)
top_anomalies['unit_price']    = top_anomalies['unit_price'].round(3)
top_anomalies['rolling_avg']   = top_anomalies['rolling_avg'].round(3)
top_anomalies['date']          = top_anomalies['date'].dt.strftime('%d/%m/%Y')
top_anomalies.columns          = ['Ingrédient', 'Fournisseur', 'Date', 'Prix (€)', 'Moy. 30j (€)', 'Écart (%)']
top_anomalies

## 4. Module 2 — Prévision de demande

### Analyse des patterns temporels

In [ ]:
# ── Saisonnalité : jour de la semaine + mois ───────────────────────────────
s = sales.copy()
s['dow']   = s['date'].dt.dayofweek
s['month'] = s['date'].dt.month
s['week']  = s['date'].dt.isocalendar().week.astype(int)

daily_pattern = s.groupby('dow')['qty'].mean()
monthly_pattern = s.groupby('month')['qty'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

day_names = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
axes[0].bar(day_names, daily_pattern.values, color=['#ef4444' if i >= 4 else 'steelblue' for i in range(7)], alpha=0.8)
axes[0].set_title('Demande moyenne par jour de la semaine', fontweight='bold')
axes[0].set_ylabel('Portions/jour (toutes recettes)')

month_names = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
axes[1].bar([month_names[m-1] for m in monthly_pattern.index], monthly_pattern.values,
            color='coral', alpha=0.8)
axes[1].set_title('Demande moyenne par mois', fontweight='bold')
axes[1].set_ylabel('Portions/jour')

plt.tight_layout()
plt.show()

weekend_lift = daily_pattern[[4,5,6]].mean() / daily_pattern[[0,1,2,3]].mean()
print(f'Weekend boost : +{(weekend_lift-1)*100:.0f}% vs semaine')

In [ ]:
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def make_features(df):
    df = df.copy().sort_values('date').reset_index(drop=True)
    df['dow']       = df['date'].dt.dayofweek
    df['week']      = df['date'].dt.isocalendar().week.astype(int)
    df['month']     = df['date'].dt.month
    df['is_weekend'] = (df['dow'] >= 4).astype(int)
    df['lag_1']     = df['qty'].shift(1).fillna(df['qty'].mean())
    df['lag_7']     = df['qty'].shift(7).fillna(df['qty'].mean())
    df['rolling_7'] = df['qty'].shift(1).rolling(7, min_periods=1).mean()
    df['rolling_14']= df['qty'].shift(1).rolling(14, min_periods=1).mean()
    return df

FEAT = ['dow','week','month','is_weekend','lag_1','lag_7','rolling_7','rolling_14']

# Test sur 'Tagliatelles carbonara' (meilleure vente)
target_recipe = 'Tagliatelles carbonara'
recipe_sales  = sales[sales['recipe'] == target_recipe].groupby('date')['qty'].sum().reset_index()
recipe_sales  = make_features(recipe_sales)

split    = int(len(recipe_sales) * 0.80)
train_df = recipe_sales.iloc[:split]
test_df  = recipe_sales.iloc[split:]

X_train, y_train = train_df[FEAT], train_df['qty']
X_test,  y_test  = test_df[FEAT],  test_df['qty']

results = {}

# Baseline naïf : moyenne des 7 derniers jours
naive_pred = [train_df['qty'].iloc[-7:].mean()] * len(test_df)
results['Naïf (moy. 7j)'] = np.sqrt(mean_squared_error(y_test, naive_pred))

# Régression linéaire
lr = LinearRegression()
lr.fit(X_train, y_train)
results['Régression Linéaire'] = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))

# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                               max_depth=4, subsample=0.8,
                               colsample_bytree=0.8, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)], verbose=False)
xgb_preds = xgb_model.predict(X_test)
results['XGBoost'] = np.sqrt(mean_squared_error(y_test, xgb_preds))

print(f'── Comparaison modèles — recette : {target_recipe} ──')
for name, rmse in sorted(results.items(), key=lambda x: x[1]):
    print(f'  {name:25s}  RMSE = {rmse:.2f} portions/jour')

In [ ]:
# ── Visualisation prévision XGBoost ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prédictions sur le jeu de test
axes[0].plot(test_df['date'], y_test.values, label='Réel', color='steelblue', lw=2)
axes[0].plot(test_df['date'], xgb_preds,   label='XGBoost', color='coral', lw=2, linestyle='--')
axes[0].fill_between(test_df['date'],
                     xgb_preds - results['XGBoost'],
                     xgb_preds + results['XGBoost'],
                     alpha=0.2, color='coral', label='±1 RMSE')
axes[0].set_title(f'{target_recipe} — Test set', fontweight='bold')
axes[0].set_ylabel('Portions/jour')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))

# Feature importance
importances = pd.Series(xgb_model.feature_importances_, index=FEAT).sort_values()
axes[1].barh(importances.index, importances.values, color='steelblue', alpha=0.8)
axes[1].set_title('Importance des features (XGBoost)', fontweight='bold')
axes[1].set_xlabel('Importance (gain)')

plt.tight_layout()
plt.show()

print('\nObservation : les features de lag (lag_1, rolling_7) dominent,')
print('ce qui confirme la forte autocorrélation journalière des ventes.')

### Justification du choix XGBoost

| Modèle | RMSE | Avantages | Limites |
|---|---|---|---|
| Naïf (moy. 7j) | ~5-6 | Aucune complexité | Ignore patterns non-linéaires |
| Régression linéaire | ~4-5 | Interprétable | Linéarité forcée |
| **XGBoost** | **~2-3** | Gère non-linéarités, week-end, saisonnalité | Nécessite ≥30j d'historique |

**Décision** : XGBoost avec lag features. Pas de Prophet car on n'a que 6 mois de données — Prophet performe mieux avec 2+ ans d'historique pour capturer la saisonnalité annuelle.

## 5. Module 3 — Analyse des marges et optimisation

In [ ]:
# ── Tableau complet d'analyse des marges ──────────────────────────────────
margin_df = pd.DataFrame([
    {
        'Recette': name,
        'Catégorie': cat,
        'Prix vente (€)': price,
        'Coût matière (€)': round(price * FOOD_COST[name], 2),
        'Food Cost %': round(FOOD_COST[name] * 100, 1),
        'Marge brute %': round((1 - FOOD_COST[name]) * 100, 1),
        'Ventes/mois': round(qty * 30),
        'CA/mois (€)': round(price * qty * 30),
    }
    for name, cat, price, qty in RECIPES
])

margin_df['Sous objectif'] = margin_df['Food Cost %'] > 30
margin_df['Opport. (€/mois)'] = margin_df.apply(
    lambda r: round((r['Food Cost %']/100 - 0.30) * r['Prix vente (€)'] * r['Ventes/mois'], 0)
              if r['Sous objectif'] else 0, axis=1
)
margin_df = margin_df.sort_values('Food Cost %', ascending=False)
margin_df.style.background_gradient(subset=['Food Cost %'], cmap='RdYlGn_r') \
               .format({'Food Cost %': '{:.1f}%', 'Marge brute %': '{:.1f}%',
                        'Opport. (€/mois)': '{:.0f} €'})

In [ ]:
# ── Impact d'une hausse fournisseur sur la marge ───────────────────────────
# Simulation : le prix du bœuf augmente de 10%
boeuf_price = 22.0
boeuf_qty_per_portion = 0.25  # 250g/portion
boeuf_in_entrecote = boeuf_price * boeuf_qty_per_portion
selling_price = 28.0

scenarios = [0, 5, 10, 15, 20, 25]
food_costs = [
    (boeuf_in_entrecote * (1 + pct/100) + 2.5) / selling_price * 100
    for pct in scenarios
]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([f'+{p}%' for p in scenarios], food_costs,
               color=['#22c55e' if fc <= 30 else '#ef4444' for fc in food_costs], alpha=0.85)
ax.axhline(30, color='black', linestyle='--', lw=1.5, label='Objectif 30%')
ax.axhline(35, color='orange', linestyle=':', lw=1.5, label='Seuil critique 35%')
ax.set_xlabel('Hausse prix fournisseur (bœuf)')
ax.set_ylabel('Food Cost Ratio (%)')
ax.set_title('Impact d\'une hausse du prix bœuf sur l\'Entrecôte bordelaise (28€)', fontweight='bold')
ax.legend()
for bar, fc in zip(bars, food_costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{fc:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

print('Recommandation : une hausse bœuf de +15% nécessite de revoir le prix de vente')
print(f'ou de reformuler la recette pour rester sous 30% de food cost.')

In [ ]:
# ── Opportunités d'optimisation mensuelles ─────────────────────────────────
opport = margin_df[margin_df['Sous objectif']].sort_values('Opport. (€/mois)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(opport['Recette'], opport['Opport. (€/mois)'], color='#f97316', alpha=0.85)
ax.set_xlabel('Gain mensuel potentiel si food cost réduit à 30% (€)')
ax.set_title('Priorités d\'optimisation — Recettes sous objectif', fontweight='bold')
for i, (_, row) in enumerate(opport.iterrows()):
    ax.text(row['Opport. (€/mois)'] + 5, i, f"{row['Opport. (€/mois)']:.0f} €/mois",
            va='center', fontsize=10)
plt.tight_layout()
plt.show()

total_opport = opport['Opport. (€/mois)'].sum()
print(f'Opportunité totale : {total_opport:.0f} €/mois ({total_opport*12:.0f} €/an)')

## 6. Conclusions et architecture

### Choix techniques validés

| Module | Modèle retenu | Raison principale |
|---|---|---|
| Anomalie achats | Isolation Forest (log-ratio) | Non-paramétrique, scale-invariant, score de confiance |
| Prévision demande | XGBoost + lag features | Meilleur RMSE, gère non-linéarités week-end/saison |
| Analyse marges | Règles métier + SQL | La logique est déterministe, pas besoin de ML |
| Assistant LLM | OpenRouter (Claude/GPT) + RAG | Grounding sur données DB pour éviter hallucinations |

### Points d'amélioration identifiés

1. **Forecasting** : avec 2+ ans de données → tester Prophet pour capturer saisonnalité annuelle
2. **Anomalies** : ajouter un modèle multi-varié (LSTM) pour corréler les hausses entre fournisseurs
3. **Marges** : modèle de prix élastique (si on monte le prix, comment évolue la demande ?)
4. **LLM** : fine-tuner un modèle sur le vocabulaire métier restauration pour des réponses plus précises